In [2]:
from collections import defaultdict
from AzgaarFunctions import *
import pandas
import json

def calculate_demand_state(self):
    out = {}
    for s in self.states:
        state_demand = {}
        for resource in RESOURCES:
            pop = s.get("rural", 0) + s.get("urban", 0)
            state_demand[resource] = pop * BASE_DEMAND[resource]
        out[s["name"]] = state_demand
    return out

data = load_from_file("ViliaFull.json")


print(calculate_demand_state(data))

{'Neutrals': {'food': 223.29589930176735, 'stone': 89.31835972070695, 'gold': 44.659179860353476}, 'Monch': {'food': 302.2046013498306, 'stone': 120.88184053993226, 'gold': 60.44092026996613}, 'Bacseslamia': {'food': 2946.7630981134175, 'stone': 1178.705239245367, 'gold': 589.3526196226835}, 'Kouria': {'food': 8101.5336991728545, 'stone': 3240.613479669142, 'gold': 1620.306739834571}, 'Memein': {'food': 2130.1737527983187, 'stone': 852.0695011193275, 'gold': 426.03475055966373}, 'Ingne': {'food': 2752.0686494166853, 'stone': 1100.827459766674, 'gold': 550.413729883337}, 'Ardonia': {'food': 1449.9021475809814, 'stone': 579.9608590323926, 'gold': 289.9804295161963}, 'Loshin': {'food': 8667.679304345727, 'stone': 3467.071721738291, 'gold': 1733.5358608691456}, 'Degyesia': {'food': 4147.284805742741, 'stone': 1658.9139222970964, 'gold': 829.4569611485482}, 'Anaia': {'food': 6252.690898558139, 'stone': 2501.076359423256, 'gold': 1250.538179711628}, 'Ogskifta': {'food': 2176.066747987747, 's

In [3]:

calculate_cell_reserve(data)
print("Total stone max:", sum(c["stone_max"] for c in data.cells))
print("stone Reserve:", sum(c["stone_reserve"] for c in data.cells))
print("Total gold max:", sum(c["gold_max"] for c in data.cells))
print("gold Reserve:", sum(c["gold_reserve"] for c in data.cells))



Total stone max: 45623.3965183501
stone Reserve: 456233.96518350096
Total gold max: 22318.90950459942
gold Reserve: 223189.0950459942


In [4]:

print(tick(data))
print(calculate_demand_state(data))



{'Neutrals': {'food': 373.2485845232009, 'stone': 64.43628807663917, 'gold': 31.001820036768912}, 'Monch': {'food': 654.4711554813383, 'stone': 156.44904204368584, 'gold': 76.33596051216124}, 'Bacseslamia': {'food': 7249.494476523395, 'stone': 663.707453635931, 'gold': 331.477784821391}, 'Kouria': {'food': 19337.44332794192, 'stone': 3991.057343716628, 'gold': 1939.7862523472336}, 'Memein': {'food': 4981.6098310554025, 'stone': 1337.6276315712942, 'gold': 656.344068765044}, 'Ingne': {'food': 6707.611832284923, 'stone': 1132.1011217415332, 'gold': 546.17720287621}, 'Ardonia': {'food': 3870.7875650924434, 'stone': 754.2777004754538, 'gold': 373.1724197575449}, 'Loshin': {'food': 20256.496756878685, 'stone': 4608.582587457596, 'gold': 2260.8315666830517}, 'Degyesia': {'food': 10530.922603585715, 'stone': 1352.37839101553, 'gold': 656.8351879727841}, 'Anaia': {'food': 15093.39989011526, 'stone': 2981.247750004529, 'gold': 1434.2660560053575}, 'Ogskifta': {'food': 5523.474478221552, 'stone'

In [5]:

print("Total stone max:", sum(c["stone_max"] for c in data.cells))
print("Stone Reserve:", sum(c["stone_reserve"] for c in data.cells))
print("Total gold max:", sum(c["gold_max"] for c in data.cells))
print("Gold Reserve:", sum(c["gold_reserve"] for c in data.cells))


Total stone max: 45623.3965183501
Stone Reserve: 410610.56866515084
Total gold max: 22318.90950459942
Gold Reserve: 200870.1855413948


In [6]:
#figure out the trade betweent burgs and their nearest city

from collections import defaultdict
graph = defaultdict(list)
def calculate_nearest_city(graph, cells):



    # 1. Normal neighbor connections
    for cell in data.cells:
        cid = cell["i"]
        for n in cell["c"]:
            graph[cid].append((n, 1.0))  # base cost

    for route in data.routes:
        group = route.get("group")

        # assign cost by type
        if group == "roads":
            road_cost = 0.3
        elif group == "trails":
            road_cost = 0.6
        else:
            road_cost = 1.0

        points = route["points"]

        for i in range(len(points) - 1):
            c1 = points[i][2]
            c2 = points[i+1][2]

            graph[c1].append((c2, road_cost))
            graph[c2].append((c1, road_cost))

    import heapq

    def compute_nearest_city(graph, cells):
        pq = []
        dist = {}
        parent = {}
        nearest_city = {}

        # initialize with all cities
        for cell in cells:
            if cell.get("burg", 0) != 0:
                cid = cell["i"]
                dist[cid] = 0
                nearest_city[cid] = cid
                parent[cid] = None
                heapq.heappush(pq, (0, cid))

        # expand outward
        while pq:
            cost, current = heapq.heappop(pq)

            for neighbor, edge_cost in graph[current]:
                new_cost = cost + edge_cost

                if neighbor not in dist or new_cost < dist[neighbor]:
                    dist[neighbor] = new_cost
                    parent[neighbor] = current
                    nearest_city[neighbor] = nearest_city[current]
                    heapq.heappush(pq, (new_cost, neighbor))

        return dist, parent, nearest_city

    def get_path(cell_id, parent):
        path = []
        while cell_id is not None:
            path.append(cell_id)
            cell_id = parent[cell_id]
        return path[::-1]

    dist, parent, nearest_city = compute_nearest_city(graph, data.cells)
    return nearest_city, dist, parent


In [9]:

def calculate_demand_cell(self):
    out = {}
    for c in self.cells:
        cell_id = c["i"]
        if c.get("pop", 0) > 0:
            out[cell_id] = {}
            for resource in RESOURCES:
                out[cell_id][resource] = c["pop"] * BASE_DEMAND[resource]
        else:
            out[cell_id] = {resource: 0 for resource in RESOURCES}

    return out

print(calculate_demand_cell(data)[0])


# now that we have the nearest city for each cell, we can calculate the surplus/deficit for each cell and see how much trade is needed to reach balance
cell_demand = calculate_demand_cell(data)



{'food': 0, 'stone': 0, 'gold': 0}


In [ ]:

def tick(self):
    # food production is based on the biome, with a bonus for being near a river or coast, and a penalty for being at high elevation
    def calculate_food_production_state():
        cell_production = {}  # state_id -> total food
        for c in self.cells:
            if c.get("h", 0) > 20:  # Only cells with elevation above 20 can produce
                biome_id = c["biome"]
                grain_yield = GRAIN_BIOME.get(biome_id, 0)
                # Bonus for being near a river or coast
                if c.get("river", False):
                    grain_yield *= 1.2
                if c.get("coast", False):
                    grain_yield *= 1.1
                if c.get("pop", 0) > 0:
                    grain_yield *= 1 + (c["pop"] / 1)  # Bonus for population

                grain_yield = grain_yield - c.get("height", 0) * 0.01  # Penalty for height
                # FIX: accumulate per state_id instead of overwriting
                state_id = c["state"]
                cell_production[state_id] = cell_production.get(state_id, 0) + grain_yield

        # Roll up cell totals to state names
        out = {}
        for state in self.states:
            state_id = state["i"]
            out[state["name"]] = {"food": cell_production.get(state_id, 0)}
        return out
    
    def calculate_stone_production_state():
        cell_production = {}  # state_id -> total stone
        for c in self.cells:
            if c.get("h", 0) > 20:  # Only cells with elevation above 120 can produce
                biome_id = c["biome"]
                stone_yield = STONE_BIOME.get(biome_id, 0)
                # Bonus for being near a river or coast
                if c.get("river", False):
                    stone_yield *= 1.2
                if c.get("coast", False):
                    stone_yield *= 0.8
                if c.get("pop", 0) > 0:
                    stone_yield *= 1 + (c["pop"] / 1)  # Bonus for population

                stone_yield *= 1 + c.get("height", 0)/1000
                reserve = c.get("stone_reserve", 0)
                actual = min(stone_yield, reserve)

                # reduce reserve
                c["stone_reserve"] = reserve - actual

                # FIX: accumulate per state_id instead of overwriting
                state_id = c["state"]
                cell_production[state_id] = cell_production.get(state_id, 0) + actual

        # Roll up cell totals to state names (FIX: no stale state_id keys to delete)
        out = {}
        for state in self.states:
            state_id = state["i"]
            out[state["name"]] = {"stone": cell_production.get(state_id, 0)}
        return out
    
    def calculate_gold_production_state():
        cell_production = {}  # state_id -> total gold
        for c in self.cells:
            if c.get("h", 0) > 20:  # Only cells with elevation above 120 can produce
                biome_id = c["biome"]
                gold_yield = GOLD_BIOME.get(biome_id, 0)
                # Bonus for being near a river or coast
                if c.get("river", False):
                    gold_yield *= 1.2
                if c.get("coast", False):
                    gold_yield *= 0.8
                if c.get("pop", 0) > 0:
                    gold_yield *= 1 + (c["pop"] / 1)  # Bonus for population

                gold_yield *= 1 + c.get("height", 0)/1000
                reserve = c.get("gold_reserve", 0)
                actual = min(gold_yield, reserve)

                # reduce reserve
                c["gold_reserve"] = reserve - actual

                # FIX: accumulate per state_id instead of overwriting
                state_id = c["state"]
                cell_production[state_id] = cell_production.get(state_id, 0) + actual

        # Roll up cell totals to state names (FIX: no stale state_id keys to delete)
        out = {}
        for state in self.states:
            state_id = state["i"]
            out[state["name"]] = {"gold": cell_production.get(state_id, 0)}
        return out

    def calcuulate_food_production_cell():
        cell_production = {}  # cell_id -> total food
        for c in self.cells:
            if c.get("h", 0) > 20:  # Only cells with elevation above 20 can produce
                biome_id = c["biome"]
                grain_yield = GRAIN_BIOME.get(biome_id, 0)
                # Bonus for being near a river or coast
                if c.get("river", False):
                    grain_yield *= 1.2
                if c.get("coast", False):
                    grain_yield *= 1.1
                if c.get("pop", 0) > 0:
                    grain_yield *= 1 + (c["pop"] / 1)  # Bonus for population

                grain_yield = grain_yield - c.get("height", 0) * 0.01  # Penalty for height
                cell_id = c["i"]
                cell_production[cell_id] = grain_yield

        return cell_production
        
    def calculate_stone_production_cell():
        cell_production = {}  # cell_id -> total stone
        for c in self.cells:
            if c.get("h", 0) > 20:  # Only cells with elevation above 120 can produce
                biome_id = c["biome"]
                stone_yield = STONE_BIOME.get(biome_id, 0)
                # Bonus for being near a river or coast
                if c.get("river", False):
                    stone_yield *= 1.2
                if c.get("coast", False):
                    stone_yield *= 0.8
                if c.get("pop", 0) > 0:
                    stone_yield *= 1 + (c["pop"] / 1)  # Bonus for population

                stone_yield *= 1 + c.get("height", 0)/1000
                reserve = c.get("stone_reserve", 0)
                actual = min(stone_yield, reserve)

                # reduce reserve
                c["stone_reserve"] = reserve - actual

                cell_id = c["i"]
                cell_production[cell_id] = actual

        return cell_production
    
    def calculate_gold_production_cell():
        cell_production = {}  # cell_id -> total gold
        for c in self.cells:
            if c.get("h", 0) > 20:  # Only cells with elevation above 120 can produce
                biome_id = c["biome"]
                gold_yield = GOLD_BIOME.get(biome_id, 0)
                # Bonus for being near a river or coast
                if c.get("river", False):
                    gold_yield *= 1.2
                if c.get("coast", False):
                    gold_yield *= 0.8
                if c.get("pop", 0) > 0:
                    gold_yield *= 1 + (c["pop"] / 1)  # Bonus for population

                gold_yield *= 1 + c.get("height", 0)/1000
                reserve = c.get("gold_reserve", 0)
                actual = min(gold_yield, reserve)

                # reduce reserve
                c["gold_reserve"] = reserve - actual

                cell_id = c["i"]
                cell_production[cell_id] = actual

        return cell_production

    state_grain = calculate_food_production_state()
    state_stone = calculate_stone_production_state()
    state_gold = calculate_gold_production_state()

    grain = calcuulate_food_production_cell()
    stone = calculate_stone_production_cell()
    gold = calculate_gold_production_cell()

    production = {
        c["i"]: {
            "food": grain.get(c["i"], 0),
            "stone": stone.get(c["i"], 0),
            "gold": gold.get(c["i"], 0),
        }
        for c in self.cells
    }

    demand = calculate_demand_cell(self)

    def calculate_cell_net(production_map, demand_map):
        net = {}
        surplus = {}
        deficit = {}
        for c in self.cells:
            cell_id = c["i"]
            net[cell_id] = {}
            surplus[cell_id] = {}
            deficit[cell_id] = {}
            for resource in RESOURCES:
                prod = production_map.get(cell_id, {}).get(resource, 0)
                dem = demand_map.get(cell_id, {}).get(resource, 0)
                diff = prod - dem
                net[cell_id][resource] = diff
                surplus[cell_id][resource] = max(diff, 0)
                deficit[cell_id][resource] = max(-diff, 0)
        return net, surplus, deficit

    def get_trade_path(cell_id, parent_map):
        path = []
        current = cell_id
        while current is not None:
            path.append(current)
            current = parent_map.get(current)
        return path[::-1]

    net, surplus, deficit = calculate_cell_net(production, demand)
    nearest_city, dist, parent = calculate_nearest_city(graph, self.cells)

    city_stock = {}
    for c in self.cells:
        cid = c["i"]
        if c.get("burg", 0) != 0:
            city_stock[cid] = surplus.get(cid, {resource: 0 for resource in RESOURCES}).copy()

    trade_log = []
    for c in self.cells:
        cid = c["i"]
        if c.get("burg", 0) != 0:
            continue

        city_id = nearest_city.get(cid)
        if city_id is None or city_id not in city_stock:
            continue

        sold = {}
        gold_balance = 0
        for resource in RESOURCES:
            if resource == "gold":
                continue
            amount = surplus[cid].get(resource, 0)
            if amount > 0:
                sold[resource] = amount
                gold_balance += amount
                city_stock[city_id]["gold"] = city_stock[city_id].get("gold", 0) + amount

        bought = {}
        for resource in [r for r in RESOURCES if r != "gold"]:
            need = deficit[cid].get(resource, 0)
            available = city_stock[city_id].get(resource, 0)
            if need <= 0 or available <= 0 or gold_balance <= 0:
                continue
            purchase = min(need, available, gold_balance)
            bought[resource] = purchase
            gold_balance -= purchase
            city_stock[city_id][resource] -= purchase
            deficit[cid][resource] -= purchase

        if sold or bought:
            trade_log.append({
                "cell_id": cid,
                "nearest_city": city_id,
                "path": get_trade_path(cid, parent),
                "sold_to_city": sold,
                "bought_from_city": bought,
                "remaining_gold_balance": gold_balance,
                "remaining_deficit": deficit[cid],
            })

    combined = {
        "production": production,
        "demand": demand,
        "net": net,
        "surplus": surplus,
        "deficit": deficit,
        "trade_log": trade_log,
        "city_stock_after_trade": city_stock,
    }

    return combined

tick(data)

{(1965, 'food'): 0.5608500242233276, (1965, 'stone'): 0.22434000968933107, (1965, 'gold'): 0.11217000484466554, (1966, 'food'): 0.4972499907016754, (1966, 'stone'): 0.19889999628067018, (1966, 'gold'): 0.09944999814033509, (1967, 'food'): 1.2424999475479126, (1967, 'stone'): 0.49699997901916504, (1967, 'gold'): 0.24849998950958252, (1968, 'food'): 0.34869998693466187, (1968, 'stone'): 0.13947999477386475, (1968, 'gold'): 0.06973999738693237, (1969, 'food'): 0.5727999806404114, (1969, 'stone'): 0.22911999225616456, (1969, 'gold'): 0.11455999612808228, (1970, 'food'): 0.243599995970726, (1970, 'stone'): 0.09743999838829041, (1970, 'gold'): 0.04871999919414521, (1971, 'food'): 0.4172999858856201, (1971, 'stone'): 0.16691999435424806, (1971, 'gold'): 0.08345999717712403, (1972, 'food'): 0.34994998574256897, (1972, 'stone'): 0.1399799942970276, (1972, 'gold'): 0.0699899971485138, (1973, 'food'): 0.3077999949455261, (1973, 'stone'): 0.12311999797821045, (1973, 'gold'): 0.061559998989105226, 

{}